# Creating a Custom TuneControl Task

This notebook demonstrates how to build a minimal task module from scratch.
We implement a mass-spring-damper system with a PD controller, expose it
as a TuneControl task, and evaluate the resulting objective.


## 1. Imports
We pull in the task base class, helper utilities, and the `register_task` decorator.


In [6]:
from dataclasses import dataclass
import torch

from tunecontrol.tasks.base import Task
from tunecontrol.tasks.module_utils import TaskConfig
from tunecontrol.tasks import register_task


## 2. Simulator
The simulator integrates a simple mass-spring-damper system controlled by
a PD controller parameterised by $\theta = [K_p, K_d]$.


In [2]:
@dataclass
class MassSpringSimulator:
    dt: float = 0.02
    horizon: int = 200
    mass: float = 1.0
    damping: float = 0.2
    stiffness: float = 1.0
    reference: float = 1.0

    def simulate(self, theta: torch.Tensor):
        Kp, Kd = float(theta[0]), float(theta[1])
        pos = torch.tensor(0.0, dtype=torch.float64)
        vel = torch.tensor(0.0, dtype=torch.float64)

        time = []
        positions = []
        velocities = []
        controls = []
        references = []

        for step in range(self.horizon):
            t = step * self.dt
            error = self.reference - pos
            control = Kp * error - Kd * vel
            accel = (control - self.damping * vel - self.stiffness * pos) / self.mass
            vel = vel + accel * self.dt
            pos = pos + vel * self.dt

            time.append(t)
            positions.append(pos.item())
            velocities.append(vel.item())
            controls.append(control.item())
            references.append(self.reference)

        trajectory = {
            "time": torch.tensor(time, dtype=torch.float64),
            "states": torch.tensor(list(zip(positions, velocities)), dtype=torch.float64),
            "control": torch.tensor(controls, dtype=torch.float64),
            "reference": torch.tensor(references, dtype=torch.float64),
        }
        return trajectory


## 3. Task wrapper
We wrap the simulator in a TuneControl `Task`, define bounds, and compute
a simple integral of squared error (with a small control penalty) as the objective.


In [3]:
class MassSpringTask(Task):
    def __init__(self, *, mass: float = 1.0, damping: float = 0.2, stiffness: float = 1.0) -> None:
        self.sim = MassSpringSimulator(mass=mass, damping=damping, stiffness=stiffness)
        self.dim = 2
        self.bounds = torch.tensor([[0.0, 0.0], [20.0, 5.0]], dtype=torch.float64).T
        self.is_minimization = True
        self.config = TaskConfig(
            name="MassSpring",
            dim=self.dim,
            bounds=self.bounds,
            is_minimization=True,
            metadata={"description": "PD-controlled mass-spring-damper"},
        )

    def _evaluate(self, theta: torch.Tensor):
        trajectory = self.sim.simulate(theta)
        position = trajectory["states"][:, 0]
        control = trajectory["control"]
        ref = trajectory["reference"]
        error = ref - position
        dt = self.sim.dt
        ise = torch.sum(error.pow(2)) * dt
        r_control = 0.01 * torch.sum(control.pow(2)) * dt
        value = ise + r_control
        info = {
            "theta": theta.detach().clone(),
            "trajectory": trajectory,
        }
        return value.to(dtype=theta.dtype), info


## 4. Register the task
Registration exposes the task via `tunecontrol.make`.


In [4]:
@register_task("mass_spring/2d/ise")
def build_mass_spring():
    return MassSpringTask()


## 5. Evaluate
Finally we instantiate the task and evaluate a sample controller.


In [ ]:
import tunecontrol as tc

task = tc.make("mass_spring/2d/ise")
theta = torch.tensor([5.0, 0.5], dtype=torch.float64)
value, info = task.evaluate(theta)
value.item(), info["trajectory"].keys()


(0.7723211844779755, dict_keys(['time', 'states', 'control', 'reference']))